# EEG_25b — Consistenza trial-to-trial: C0 dinamico vs C1 statico?

**Ipotesi** (da [[EEG_25]]): la proficiency di C0 è uno **stato** (reliability bAcc ≈0),
quella di C1 un **tratto** (reliability 0.45). Predizione diretta e testabile:

> La firma neurale di C0 dovrebbe essere **meno consistente trial-per-trial** di quella di C1.

**Metodo**: per ogni soggetto calcolo la similarità media tra i suoi trial (mean inter-trial
pattern correlation) su due feature: alpha power (61-dim, la firma C0 di EEG_22) e abs_pcc
(1830-dim, la base dei fenotipi). Indice alto = trial simili tra loro = firma **statica**.
Poi confronto C0 vs C1 (Mann-Whitney + Cohen's d).

**Predizione**: consistency(C1) > consistency(C0).  CoV(C0) > CoV(C1).


In [ ]:
import json, warnings
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.signal import welch
from scipy.stats import mannwhitneyu
warnings.filterwarnings('ignore')

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_ROOT = project_root / 'data' / 'hypergraphs_pruned_abs_pcc'
CLUSTER_JSON = project_root / 'configs' / 'eeg16b_cluster_labels.json'
FIG_DIR = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)
CACHE = project_root / 'data' / 'eeg25b_pertrial_feats.pkl'

FS = 256
ALPHA = (8, 13)
MAX_TRIALS = 250          # subsample per soggetto (l'indice converge ben prima)
SEED = 0
N_CHANNELS = 61
PAIR_ROWS, PAIR_COLS = np.triu_indices(N_CHANNELS, k=1)  # 1830 coppie

cd = json.loads(CLUSTER_JSON.read_text())
SUBJ_CLUSTER = {int(s): int(l) for s, l in zip(cd['subj_ids'], cd['labels'])}
print('cluster labels:', sum(v==0 for v in SUBJ_CLUSTER.values()), 'C0,',
      sum(v==1 for v in SUBJ_CLUSTER.values()), 'C1')
print('DATA_ROOT esiste:', DATA_ROOT.exists())


## §2 — Estrazione feature per-trial (alpha 61 + PCC 1830)

In [ ]:
import pickle
rng = np.random.default_rng(SEED)

def per_trial_feats(sid):
    """Ritorna (F_alpha: n×61, F_pcc: n×1830) per il soggetto, subsample MAX_TRIALS."""
    paths = []
    for sess_dir in sorted(DATA_ROOT.glob(f'P{sid:03d}_S*')):
        paths += sorted(sess_dir.glob('trial_*.pt'))
    if not paths:
        return None, None
    if len(paths) > MAX_TRIALS:
        idx = rng.choice(len(paths), MAX_TRIALS, replace=False)
        paths = [paths[i] for i in sorted(idx)]
    A, P = [], []
    for p in paths:
        x = torch.load(p, weights_only=False)['x'].float().numpy()  # (61,384)
        f, psd = welch(x, fs=FS, nperseg=128, axis=1)               # (61, nf)
        amask = (f >= ALPHA[0]) & (f <= ALPHA[1])
        A.append(psd[:, amask].mean(1))                             # (61,)
        corr = np.abs(np.corrcoef(x))                               # (61,61)
        P.append(corr[PAIR_ROWS, PAIR_COLS])                        # (1830,)
    return np.array(A), np.array(P)

if CACHE.exists():
    with open(CACHE, 'rb') as fh:
        FEATS = pickle.load(fh)
    print('cache caricata:', len(FEATS), 'soggetti')
else:
    FEATS = {}
    for sid in sorted(SUBJ_CLUSTER):
        Fa, Fp = per_trial_feats(sid)
        if Fa is None or len(Fa) < 20:
            continue
        FEATS[sid] = {'alpha': Fa, 'pcc': Fp}
    with open(CACHE, 'wb') as fh:
        pickle.dump(FEATS, fh)
    print('estratti e salvati:', len(FEATS), 'soggetti')


## §3 — Indice di consistenza trial-to-trial

`consistency` = media off-diagonal di corrcoef(F) (righe = trial). Alto = trial simili (statico).
`cov` = media su feature di std/|mean| tra trial. Alto = variabile (dinamico).

In [ ]:
def consistency(F):
    C = np.corrcoef(F)                 # (n_trials × n_trials) pattern similarity
    iu = np.triu_indices_from(C, k=1)
    return float(np.nanmean(C[iu]))

def cov_index(F):
    m = np.abs(F.mean(0)) + 1e-9
    return float(np.nanmean(F.std(0) / m))

ROWS = []
for sid, d in FEATS.items():
    cl = SUBJ_CLUSTER.get(sid)
    if cl not in (0, 1):
        continue
    ROWS.append({
        'sid': sid, 'cluster': cl, 'n': len(d['alpha']),
        'cons_alpha': consistency(d['alpha']),
        'cons_pcc':   consistency(d['pcc']),
        'cov_alpha':  cov_index(d['alpha']),
    })

import pandas as pd
df = pd.DataFrame(ROWS)
print(df.groupby('cluster')[['cons_alpha','cons_pcc','cov_alpha','n']].mean().round(4))


## §4 — Test C0 vs C1 + figura

In [ ]:
def compare(col, higher_in='C1'):
    a = df[df.cluster==0][col].values
    b = df[df.cluster==1][col].values
    U, p = mannwhitneyu(a, b, alternative='two-sided')
    # Cohen's d
    pooled = np.sqrt(((len(a)-1)*a.var(ddof=1) + (len(b)-1)*b.var(ddof=1)) / (len(a)+len(b)-2))
    d = (b.mean() - a.mean()) / (pooled + 1e-9)   # +d = C1 > C0
    return a.mean(), b.mean(), p, d

print('='*64)
print(f'{"feature":14s} {"C0":>8s} {"C1":>8s} {"p":>9s} {"d(C1-C0)":>9s}')
print('-'*64)
for col in ['cons_alpha','cons_pcc','cov_alpha']:
    m0,m1,p,d = compare(col)
    print(f'{col:14s} {m0:8.4f} {m1:8.4f} {p:9.4f} {d:+9.3f}')
print('='*64)

fig, axes = plt.subplots(1, 3, figsize=(15,5)); fig.patch.set_facecolor('white')
COL = {0:'#4A90E2', 1:'#FF8C42'}
for ax, col, title in zip(axes,
        ['cons_alpha','cons_pcc','cov_alpha'],
        ['Consistenza ALPHA (statico↑)','Consistenza PCC (statico↑)','CoV ALPHA (dinamico↑)']):
    data = [df[df.cluster==0][col].values, df[df.cluster==1][col].values]
    bp = ax.boxplot(data, labels=['C0','C1'], patch_artist=True, widths=0.6)
    for patch, c in zip(bp['boxes'], [COL[0], COL[1]]):
        patch.set_facecolor(c); patch.set_alpha(0.5)
    for i,(arr,c) in enumerate(zip(data,[COL[0],COL[1]]), start=1):
        ax.scatter(np.random.normal(i,0.05,len(arr)), arr, c=c, s=22, edgecolors='#222', lw=0.5, zorder=3)
    m0,m1,p,d = compare(col)
    ax.set_title(f'{title}\np={p:.3f}  d={d:+.2f}', fontsize=11, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
plt.suptitle('EEG_25b — Consistenza trial-to-trial: C0 vs C1', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
sp = FIG_DIR / 'eeg25b_consistency.png'
plt.savefig(sp, dpi=150, bbox_inches='tight', facecolor='white'); plt.show()
print('Salvato', sp)


## §5 — Verdetto

In [ ]:
print('='*64)
print('EEG_25b — VERDETTO: C0 dinamico vs C1 statico?')
print('='*64)
m0a,m1a,pa,da = compare('cons_alpha')
m0p,m1p,pp,dp = compare('cons_pcc')
m0c,m1c,pc,dc = compare('cov_alpha')

print(f'\nConsistenza ALPHA: C0={m0a:.3f} vs C1={m1a:.3f}  (d={da:+.2f}, p={pa:.3f})')
print(f'Consistenza PCC:   C0={m0p:.3f} vs C1={m1p:.3f}  (d={dp:+.2f}, p={pp:.3f})')
print(f'CoV ALPHA:         C0={m0c:.3f} vs C1={m1c:.3f}  (d={dc:+.2f}, p={pc:.3f})')

conf_alpha = (da > 0) and (pa < 0.05)
conf_pcc   = (dp > 0) and (pp < 0.05)
conf_cov   = (dc < 0) and (pc < 0.05)
n_conf = sum([conf_alpha, conf_pcc, conf_cov])

print('\nIpotesi "C1 statico > C0 dinamico":')
print(f'  consistency alpha C1>C0: {"SI" if conf_alpha else "no"}')
print(f'  consistency pcc   C1>C0: {"SI" if conf_pcc else "no"}')
print(f'  CoV alpha         C0>C1: {"SI" if conf_cov else "no"}')
print(f'  -> {n_conf}/3 misure confermano')
if n_conf >= 2:
    print('  VERDETTO: ipotesi SUPPORTATA - C0 piu dinamico, C1 piu statico')
elif n_conf == 0:
    print('  VERDETTO: ipotesi NON supportata - consistenza simile o opposta')
else:
    print('  VERDETTO: misto - serve approfondire')

print('\nCAVEAT: la consistenza puo essere confusa con la qualita del segnale')
print('(soggetto rumoroso = bassa consistenza). Controllare vs SNR/ampiezza.')
print('='*64)


## §6 — Note

- **Cosa testa**: se la firma neurale di C0 fluttua piu di quella di C1 tra trial → conferma diretta
  che C0 e dinamico (stato) e C1 statico (tratto), indipendente dalla bAcc.
- **Se confermato**: giustifica per C0 l'analisi a livello di trial (consistenza/microstati/HMM)
  invece che a livello di soggetto. Prossimo: predizione decodabilita trial-level WITHIN C0.
- **Caveat SNR**: aggiungere come controllo l'ampiezza/varianza media del segnale per soggetto e
  verificare che la differenza di consistenza non sia solo qualita.


## §7 — cons_pcc vs bAcc: la consistenza spaziale predice la proficiency?

In [ ]:
from scipy.stats import spearmanr

ranking = pd.read_csv(project_root / 'figures' / 'eeg13b_subject_ranking.csv')
ranking['sid'] = ranking['Subject'].str[1:].astype(int)   # P070 -> 70
bacc_map = dict(zip(ranking['sid'], ranking['Test bAcc']))

df['bacc'] = df['sid'].map(bacc_map)
df_ok = df.dropna(subset=['bacc'])

fig, axes = plt.subplots(1, 2, figsize=(12, 5)); fig.patch.set_facecolor('white')
for ax, cl, label, color in zip(axes, [0, 1], ['C0 Fronto-motor', 'C1 Fronto-occipital'], ['#4A90E2', '#FF8C42']):
    sub = df_ok[df_ok.cluster == cl]
    rho, p = spearmanr(sub['cons_pcc'], sub['bacc'])
    ax.scatter(sub['cons_pcc'], sub['bacc'], c=color, s=50, edgecolors='#222', lw=0.5, zorder=3)
    for _, row in sub.iterrows():
        if row['bacc'] > 0.33 or row['bacc'] < 0.18:
            ax.annotate(f"P{row['sid']:03d}", (row['cons_pcc'], row['bacc']),
                        fontsize=8, ha='left', va='bottom')
    z = np.polyfit(sub['cons_pcc'], sub['bacc'], 1)
    xs = np.linspace(sub['cons_pcc'].min(), sub['cons_pcc'].max(), 100)
    ax.plot(xs, np.polyval(z, xs), '--', color=color, lw=1.5)
    ax.set_xlabel('cons_pcc'); ax.set_ylabel('bAcc S-Spec')
    ax.set_title(f"{label}
Spearman rho={rho:+.3f}  p={p:.3f}", fontweight='bold')
    ax.grid(alpha=0.3)
    print(f"{label}: rho={rho:+.3f}  p={p:.3f}  n={len(sub)}")

plt.suptitle('§7 — cons_pcc vs bAcc S-Spec per cluster', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg25b_pcc_vs_bacc.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()


## §8 — Within-word vs between-word cons_pcc

Domanda: la consistenza spaziale di C1 è word-specific (Scenario A) o word-generic (Scenario B)?
- **Scenario A**: trial della stessa parola si assomigliano piu di trial di parole diverse → within > between
- **Scenario B**: tutti i trial si assomigliano ugualmente → within ≈ between (e alta cons_pcc è solo impronta soggetto)


In [ ]:
CACHE_LBL = project_root / 'data' / 'eeg25b_labeled_feats.pkl'
MAX_TRIALS_LBL = 550  # tutti i trial (110 parole × 5 sessioni = 550)

def per_trial_feats_labeled(sid):
    """Ritorna (F_pcc: n×1830, labels: n) con label parola per ogni trial."""
    paths = []
    for sess_dir in sorted(DATA_ROOT.glob(f'P{sid:03d}_S*')):
        paths += sorted(sess_dir.glob('trial_*.pt'))
    if not paths:
        return None, None
    P, L = [], []
    for p in paths:
        d = torch.load(p, weights_only=False)
        x = d['x'].float().numpy()       # (61,384)
        y = int(d['y'].item())           # label parola 0-109
        corr = np.abs(np.corrcoef(x))
        P.append(corr[PAIR_ROWS, PAIR_COLS])  # (1830,)
        L.append(y)
    return np.array(P), np.array(L)

if CACHE_LBL.exists():
    with open(CACHE_LBL, 'rb') as fh:
        FEATS_LBL = pickle.load(fh)
    print('cache labeled caricata:', len(FEATS_LBL), 'soggetti')
else:
    FEATS_LBL = {}
    for sid in sorted(SUBJ_CLUSTER):
        Fp, Ly = per_trial_feats_labeled(sid)
        if Fp is None or len(Fp) < 20:
            continue
        FEATS_LBL[sid] = {'pcc': Fp, 'labels': Ly}
    with open(CACHE_LBL, 'wb') as fh:
        pickle.dump(FEATS_LBL, fh)
    print('estratti e salvati:', len(FEATS_LBL), 'soggetti')

def within_between_delta(Fp, Ly):
    """Ritorna (within_mean, between_mean, delta=within-between)."""
    C = np.corrcoef(Fp)  # (n,n)
    within, between = [], []
    for i in range(len(Ly)):
        for j in range(i+1, len(Ly)):
            if Ly[i] == Ly[j]:
                within.append(C[i,j])
            else:
                between.append(C[i,j])
    w = float(np.nanmean(within))
    b = float(np.nanmean(between))
    return w, b, w - b

WB_ROWS = []
for sid, d in FEATS_LBL.items():
    cl = SUBJ_CLUSTER.get(sid)
    if cl not in (0, 1):
        continue
    w, b, delta = within_between_delta(d['pcc'], d['labels'])
    WB_ROWS.append({'sid': sid, 'cluster': cl, 'within': w, 'between': b, 'delta': delta})
df_wb = pd.DataFrame(WB_ROWS)

print('
=== Within vs Between word PCC ===')
for cl, name in [(0,'C0 Fronto-motor'), (1,'C1 Fronto-occipital')]:
    sub = df_wb[df_wb.cluster==cl]
    print(f"{name}: within={sub.within.mean():.4f}  between={sub.between.mean():.4f}  delta={sub.delta.mean():+.4f}  (delta>0 = word-specific)")

# t-test delta vs 0 per cluster
from scipy.stats import ttest_1samp
for cl, name in [(0,'C0'), (1,'C1')]:
    sub = df_wb[df_wb.cluster==cl]['delta'].values
    t, p = ttest_1samp(sub, 0)
    print(f"{name} delta vs 0: t={t:+.3f}  p={p:.4f}  -> {'word-SPECIFIC' if p<0.05 and t>0 else 'word-GENERIC' if p<0.05 and t<0 else 'ambiguo'}")

fig, ax = plt.subplots(figsize=(8,5)); fig.patch.set_facecolor('white')
COL2 = {0:'#4A90E2', 1:'#FF8C42'}
for cl, name in [(0,'C0'), (1,'C1')]:
    sub = df_wb[df_wb.cluster==cl]['delta'].values
    ax.scatter([name]*len(sub), sub, c=COL2[cl], s=40, alpha=0.6, edgecolors='#222', lw=0.5)
ax.axhline(0, color='red', ls='--', lw=1.5, label='delta=0 (word-generic)')
ax.set_ylabel('delta = within - between word PCC')
ax.set_title('§8 — Consistenza word-specific vs word-generic', fontweight='bold')
ax.grid(axis='y', alpha=0.3); ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg25b_within_between.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()


## §9 — P070 vs P069: cosa distingue il top dal bottom in C1?

In [ ]:
# Entrambi C1. P070=best (0.393), P069=worst (0.149)
PAIR_SIDS = {70: 'P070 (best C1, bAcc=0.393)', 69: 'P069 (worst C1, bAcc=0.149)'}

print('=== Confronto P070 vs P069 (entrambi C1) ===')
for sid, label in PAIR_SIDS.items():
    if sid not in FEATS:
        print(f"{label}: non in cache"); continue
    d = FEATS[sid]
    cp = consistency(d['pcc'])
    ca = consistency(d['alpha'])
    cv = cov_index(d['alpha'])
    n  = len(d['alpha'])
    print(f"  {label}")
    print(f"    cons_pcc={cp:.4f}  cons_alpha={ca:.4f}  cov_alpha={cv:.4f}  n_trials={n}")

# Distribuzione PCC trial-to-trial per i due soggetti
fig, axes = plt.subplots(1, 2, figsize=(12,4)); fig.patch.set_facecolor('white')
for ax, (sid, label) in zip(axes, PAIR_SIDS.items()):
    if sid not in FEATS:
        ax.set_title(f"{label}: no data"); continue
    Fp = FEATS[sid]['pcc']
    C  = np.corrcoef(Fp)
    iu = np.triu_indices_from(C, k=1)
    vals = C[iu]
    ax.hist(vals, bins=60, color='#FF8C42' if sid==70 else '#888', alpha=0.7, edgecolor='white')
    ax.axvline(float(np.nanmean(vals)), color='red', ls='--', lw=1.5,
               label=f'mean={float(np.nanmean(vals)):.3f}')
    ax.set_xlabel('PCC inter-trial'); ax.set_ylabel('count')
    ax.set_title(label, fontweight='bold'); ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('§9 — Distribuzione similarità inter-trial PCC: P070 vs P069', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg25b_p070_vs_p069.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
